# Duke Experiment: Ablation Study + Full Baseline Comparison

**Scene:** Duke — single transmitter, same zone as motivating example  
**Frequency:** 3.5 GHz

**Part 1 — Ablation:** Vary `num_sample_points` across gradient baselines to characterize the trade-off between:
- Convergence speed (iterations to threshold)
- Loss curve smoothness / gradient variance
- Total wall-clock runtime

**Part 2 — Full comparison:** Run all 7 baselines at the best sample count identified above.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCENE_XML_PATH = "../scene/scenes/Duke/scene.xml"
CARRIER_HZ     = 3.5e9

# Transmitter (same building and zone as motivating_example)
TX_NAME        = "gnb"
TX_BUILDING_ID = 33
TX_HEIGHT_M    = 10.0

# ── ZONE CONFIGURATION ──────────────────────────────────────────────────────
# Coordinates are scene-local meters. Run the visualization cell to verify.
ZONE_CENTER   = [-200.0, 300.0]
ZONE_WIDTH_M  = 200.0
ZONE_HEIGHT_M = 200.0

MAP_CONFIG = {
    'center':        [0.0, 0.0, 0.0],
    'size':          [1400, 1400],
    'cell_size':     (0.5, 0.5),
    'ground_height': 0.0,
}

# Experiment
NOISE_POWER   = 1e-10
JITTER_SEED   = 42
JITTER_MAG    = 1e-4          # Degrees — convergence tolerance
NUM_ITERATIONS = 50           # Gradient optimizer iterations

# Ablation
SAMPLE_COUNTS  = [32, 64, 128, 256, 512]
GRAD_BASELINES = ["grad_full_rejection", "grad_full_triangulated", "grad_proportional"]

# Full comparison
ALL_BASELINES  = [
    "grad_full_rejection", "grad_full_triangulated", "grad_proportional",
    "random_search", "pso", "coordinate_descent", "uma_naive",
]
OUTPUT_PATH = "../scripts/report/duke_experiment_results.json"

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import mitsuba as mi

try:
    import sionna.rt
except ImportError:
    os.system("pip install sionna-rt")
    import sionna.rt

from sionna.rt import load_scene, AntennaArray
from sionna.rt.antenna_pattern import antenna_pattern_registry

from scene_parser import extract_building_info
from tx_placement import TxPlacement
from boresight_pathsolver import create_zone_mask
from angle_utils import compute_initial_angles_from_position, azimuth_elevation_to_yaw_pitch
from multi_tx_optimizer import TxConfig
from experiment_runner import (
    ExperimentConfig, run_experiment_suite,
    compare_all_results, plot_cdf, plot_loss_curves, plot_metric_bars,
)

scene = load_scene(SCENE_XML_PATH)
scene.frequency = CARRIER_HZ

single_el = np.array([[0.0, 0.0, 0.0]])
scene.tx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("tr38901")(polarization="V"),
    normalized_positions=single_el.T,
)
scene.rx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("iso")(polarization="V"),
    normalized_positions=single_el.T,
)
for rm in scene.radio_materials.values():
    rm.scattering_coefficient = 0.4

building_info = extract_building_info(SCENE_XML_PATH, verbose=False)
print(f"Scene loaded  |  {CARRIER_HZ/1e9:.1f} GHz")

In [ ]:
# ── ZONE MASK ────────────────────────────────────────────────────────────────
zone_params = {'center': ZONE_CENTER, 'width': ZONE_WIDTH_M, 'height': ZONE_HEIGHT_M}

zone_mask, look_at_pos, zone_stats = create_zone_mask(
    map_config=MAP_CONFIG,
    zone_type='box',
    zone_params=zone_params,
    target_height=1.5,
    scene_xml_path=SCENE_XML_PATH,
    exclude_buildings=True,
)
zone_masks = {TX_NAME: zone_mask}
print(f"Zone cells: {zone_stats['num_cells']}  |  centroid: {zone_stats['centroid_xy']}")

In [ ]:
# ── TX PLACEMENT ─────────────────────────────────────────────────────────────
tx_placer = TxPlacement(scene, TX_NAME, SCENE_XML_PATH, TX_BUILDING_ID, offset=TX_HEIGHT_M)
tx_placer.set_rooftop_zone_facing(zone_stats["centroid_xy"])

tx = scene.get(TX_NAME)
tx_pos = tx.position.numpy().flatten().tolist()
print(f"TX at ({tx_pos[0]:.2f}, {tx_pos[1]:.2f}, {tx_pos[2]:.2f})")

In [ ]:
# ── INITIAL JITTER ───────────────────────────────────────────────────────────
# Same fixed jitter across all ablation runs for fair comparison.
base_az, base_el = compute_initial_angles_from_position(tx_pos, zone_stats["look_at_xyz"])

rng = np.random.default_rng(JITTER_SEED)
initial_az = base_az + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
initial_el = base_el + float(rng.uniform(-JITTER_MAG, JITTER_MAG))

yaw_r, pitch_r = azimuth_elevation_to_yaw_pitch(initial_az, initial_el)
tx.orientation = mi.Point3f(yaw_r, pitch_r, 0.0)

# Store initial state for restoration between ablation runs
_init_pos_list = tx_pos[:]
_init_yaw, _init_pitch = yaw_r, pitch_r

print(f"Initial Az = {initial_az:.6f}°,  El = {initial_el:.6f}°")

In [ ]:
# ── ZONE VISUALIZATION ───────────────────────────────────────────────────────
_cx, _cy, _ = MAP_CONFIG['center']
_w, _h = MAP_CONFIG['size']
extent = [_cx - _w/2, _cx + _w/2, _cy - _h/2, _cy + _h/2]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.ma.masked_where(zone_mask == 0, zone_mask),
          origin='lower', extent=extent, cmap='Blues', vmin=0, vmax=1, alpha=0.5)
for bdata in building_info.values():
    verts = bdata['vertices'][:, :2]
    ax.add_patch(MplPolygon(verts, closed=True, facecolor='gray',
                            edgecolor='black', linewidth=0.8, alpha=0.4))
ax.plot(tx_pos[0], tx_pos[1], '^', color='red', markersize=14,
        markeredgecolor='black', label=f'TX: {TX_NAME} (bldg {TX_BUILDING_ID})', zorder=5)
ax.plot(*zone_stats['centroid_xy'], 'o', color='steelblue', markersize=10,
        markeredgecolor='black', label='Zone centroid', zorder=5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Coverage Zone — Duke Experiment (3.5 GHz)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 1: Sample Count Ablation

Runs the three gradient baselines at each sample count in `SAMPLE_COUNTS`.
Results are stored in `ablation_results[n]` for plotting.

In [ ]:
# ── CONVERGENCE HELPER ───────────────────────────────────────────────────────
def check_convergence(suite_output, baseline_id, tx_name, tol=1e-4):
    entry = suite_output["results"].get(baseline_id, {})
    opt   = entry.get("optimizer_result")
    if not isinstance(opt, dict):
        return {"status": "error"}
    tx_res  = opt.get(tx_name, {})
    az_hist = tx_res.get("az_history", [])
    el_hist = tx_res.get("el_history", [])
    if len(az_hist) > 1:
        for i in range(1, len(az_hist)):
            if abs(az_hist[i] - az_hist[i-1]) < tol and abs(el_hist[i] - el_hist[i-1]) < tol:
                return {"status": "converged", "iteration": i}
        return {"status": "no convergence", "iterations": len(az_hist)}
    init_angles = suite_output["metadata"]["initial_angles"].get(tx_name, [None, None])
    best_angles = tx_res.get("best_angles", [None, None])
    if None in list(init_angles) + list(best_angles):
        return {"status": "error"}
    d_az = abs(best_angles[0] - init_angles[0])
    d_el = abs(best_angles[1] - init_angles[1])
    label = "no movement" if d_az < tol and d_el < tol else "displaced"
    return {"status": label, "Δaz_deg": round(d_az, 4), "Δel_deg": round(d_el, 4)}

In [ ]:
# ── ABLATION LOOP ────────────────────────────────────────────────────────────
ablation_results = {}

for n in SAMPLE_COUNTS:
    print(f"\n{'='*60}")
    print(f"  num_sample_points = {n}")
    print(f"{'='*60}")

    # Restore TX to jittered starting state before each suite call
    tx.position    = mi.Point3f(*_init_pos_list)
    tx.orientation = mi.Point3f(_init_yaw, _init_pitch, 0.0)

    tx_cfg_n = TxConfig(
        name=TX_NAME,
        building_id=TX_BUILDING_ID,
        zone_params=zone_params,
        tx_height_offset=TX_HEIGHT_M,
        num_sample_points=n,
    )
    exp_n = ExperimentConfig(
        baselines=GRAD_BASELINES,
        noise_power=NOISE_POWER,
        num_iterations=NUM_ITERATIONS,
        verbose=False,
    )
    ablation_results[n] = run_experiment_suite(
        scene=scene,
        tx_configs=[tx_cfg_n],
        map_config=MAP_CONFIG,
        scene_xml_path=SCENE_XML_PATH,
        zone_masks=zone_masks,
        exp_config=exp_n,
    )
    elapsed = {b: ablation_results[n]["results"][b]["elapsed_s"] for b in GRAD_BASELINES}
    print(f"  Elapsed: { {b: f'{v:.1f}s' for b, v in elapsed.items()} }")

In [ ]:
# ── ABLATION PLOTS ───────────────────────────────────────────────────────────
# Colors match experiment_runner.py palette
_colors = {
    "grad_full_rejection":    "#1f77b4",
    "grad_full_triangulated": "#ff7f0e",
    "grad_proportional":      "#2ca02c",
}
_labels = {
    "grad_full_rejection":    "Rejection, Full",
    "grad_full_triangulated": "Triangulated, Full",
    "grad_proportional":      "Triangulated, Prop.",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Loss curves for each sample count (one subplot per baseline would be busy;
#          show all baselines at best_n — after selecting it below)
ax1 = axes[0]
for n in SAMPLE_COUNTS:
    sr = ablation_results[n]
    for bid in GRAD_BASELINES:
        loss = sr["results"][bid].get("optimizer_result", {}).get("joint", {}).get("loss_history", [])
        if loss:
            ax1.plot(range(1, len(loss)+1), loss,
                     color=_colors[bid], alpha=0.4 + 0.15 * SAMPLE_COUNTS.index(n),
                     label=f"{_labels[bid]} n={n}" if SAMPLE_COUNTS.index(n) == len(SAMPLE_COUNTS)//2 else "_")
ax1.set_xlabel("Iteration"); ax1.set_ylabel("SIR Loss")
ax1.set_title("Loss Curves (all sample counts)")
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=6)

# Panel 2: Total runtime vs sample count
ax2 = axes[1]
x = np.arange(len(SAMPLE_COUNTS))
w = 0.25
for i, bid in enumerate(GRAD_BASELINES):
    runtimes = [ablation_results[n]["results"][bid]["elapsed_s"] for n in SAMPLE_COUNTS]
    ax2.bar(x + i*w, runtimes, width=w, color=_colors[bid], label=_labels[bid])
ax2.set_xticks(x + w)
ax2.set_xticklabels([str(n) for n in SAMPLE_COUNTS])
ax2.set_xlabel("num_sample_points"); ax2.set_ylabel("Runtime (s)")
ax2.set_title("Runtime vs. Sample Count")
ax2.legend(fontsize=7); ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: Gradient variance proxy — rolling std of loss_history
ax3 = axes[2]
WIN = 5
for n in SAMPLE_COUNTS:
    sr = ablation_results[n]
    for bid in GRAD_BASELINES:
        loss = sr["results"][bid].get("optimizer_result", {}).get("joint", {}).get("loss_history", [])
        if len(loss) >= WIN:
            rolling_std = [np.std(loss[max(0,i-WIN):i+1]) for i in range(len(loss))]
            ax3.plot(range(1, len(rolling_std)+1), rolling_std,
                     color=_colors[bid], alpha=0.4 + 0.15 * SAMPLE_COUNTS.index(n))
ax3.set_xlabel("Iteration"); ax3.set_ylabel(f"Rolling Std (window={WIN})")
ax3.set_title("Gradient Variance Proxy")
ax3.grid(True, alpha=0.3)

plt.suptitle("Ablation: num_sample_points — Duke, 3.5 GHz", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── CONVERGENCE TABLE ────────────────────────────────────────────────────────
rows = []
for n in SAMPLE_COUNTS:
    row = {"num_samples": n}
    for bid in GRAD_BASELINES:
        conv = check_convergence(ablation_results[n], bid, TX_NAME, tol=JITTER_MAG)
        row[bid] = f"iter {conv['iteration']}" if conv["status"] == "converged" else conv["status"]
    rows.append(row)

conv_df = pd.DataFrame(rows).set_index("num_samples")
conv_df.columns = [_labels[b] for b in GRAD_BASELINES]
print("\nConvergence Table (tol = 1e-4 deg):")
print(conv_df.to_string())

In [ ]:
# ── SELECT BEST SAMPLE COUNT ─────────────────────────────────────────────────
# Review the ablation plots above, then set best_n here.
best_n = 256   # ← adjust after reviewing plots

## Part 2: Full Baseline Comparison

Runs all 7 baselines at `best_n` sample points.

In [ ]:
# ── FULL COMPARISON RUN ──────────────────────────────────────────────────────
# Restore TX to jittered starting state
tx.position    = mi.Point3f(*_init_pos_list)
tx.orientation = mi.Point3f(_init_yaw, _init_pitch, 0.0)

tx_config_full = TxConfig(
    name=TX_NAME,
    building_id=TX_BUILDING_ID,
    zone_params=zone_params,
    tx_height_offset=TX_HEIGHT_M,
    num_sample_points=best_n,
)
exp_config_full = ExperimentConfig(
    baselines=ALL_BASELINES,
    noise_power=NOISE_POWER,
    num_iterations=NUM_ITERATIONS,
    output_path=OUTPUT_PATH,
)
suite_output = run_experiment_suite(
    scene=scene,
    tx_configs=[tx_config_full],
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    zone_masks=zone_masks,
    exp_config=exp_config_full,
)

In [ ]:
# ── CONVERGENCE + RESULTS TABLE ──────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"  Convergence Analysis  (tol = {JITTER_MAG:.0e} deg, n = {best_n})")
print(f"{'─'*55}")
for bid in ALL_BASELINES:
    conv = check_convergence(suite_output, bid, TX_NAME, tol=JITTER_MAG)
    print(f"  {bid:<30s}  {conv}")

df = compare_all_results(suite_output)

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_metric_bars(
    suite_output,
    metrics=("rsrp_mean_dbm", "rsrp_p10_dbm", "sir_median_db", "sir_p10_db", "coverage_fraction"),
    ax=axes[0],
)
axes[0].set_title(f"Metric Comparison (n={best_n})")

plot_loss_curves(suite_output, ax=axes[1])
axes[1].set_title("Gradient Baseline Convergence")

plot_cdf(suite_output, metric="rsrp_values_dbm", ax=axes[2])
axes[2].set_title("RSRP CDF")

plt.suptitle(f"Duke Experiment — All Baselines, n={best_n}, 3.5 GHz", y=1.02)
plt.tight_layout()
plt.show()